# Comprehensive Hyperparameter Tuning — Single-Pulse Autoencoders

This notebook performs an extensive random search comparing **two autoencoder architectures** on single waveforms:

1. **Dense (fully-connected) autoencoder** — flat layers, no spatial structure
2. **1D Convolutional autoencoder** — exploits the temporal structure of pulses

Both must do two things: reconstruct the waveform and provide a latent feature vector that linearly separates gammas from neutrons via a downstream logistic regression.

## Hyperparameters explored

**Shared (both architectures):**
- `model_type` — `"dense"` or `"cnn"`
- `latent_dim` — bottleneck size
- `activation`, `output_activation`
- `dropout`, `l2_reg`, `batch_norm`, `noise_std`
- `optimizer`, `learning_rate`, `batch_size`, `loss`

**Dense-only:**
- `n_layers` — encoder depth
- `start_width` — first encoder layer width
- `width_strategy` — `geometric`, `linear`, or `funnel`

**CNN-only:**
- `n_conv_layers` — number of stride-2 Conv1D layers (each halves the time axis)
- `kernel_size` — convolution kernel width (3, 5, or 7)
- `n_filters_start` — channel count of the first conv layer
- `filter_strategy` — `double` (16→32→64) or `constant`

## Per-trial evaluation

For every configuration we record:
1. `best_val_loss` — validation reconstruction loss
2. `clf_accuracy` — accuracy of a logistic regression on the encoder's latent features
3. `photon_err_pct` / `neutron_err_pct` — per-class error rates
4. `composite_score` — combined ranking metric (lower is better)

Hyperparameters that don't apply to a given architecture are stored as `None` in the results CSV, so the analysis code filters by `model_type` before reporting per-hyperparameter effects.

In [ ]:
import itertools
import random
import time

import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, Model, regularizers

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.metrics import accuracy_score, confusion_matrix

import matplotlib.pyplot as plt

print(f"TensorFlow {tf.__version__}")
print(f"GPUs: {tf.config.list_physical_devices('GPU')}")

## Load and split data

Same pipeline as the baseline autoencoder notebook: load Euclidean-normalized
waveforms, split 60/20/20 train/val/test, then min-max normalize to [0, 1]
(matching the sigmoid output activation).

In [ ]:
# Load X_euclidean (already L2-normalized per waveform by preprocess.py).
# No further scaling -- pure L2 is the normalization we're testing.
d = np.load("processed_waveforms.npz")
X = d["X_euclidean"].astype(np.float32)
y = d["y"].astype(np.int32)  # 0=photon, 1=neutron

# 60/20/20 train/val/test split
X_tv, X_test, y_tv, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
X_train, X_val, y_train, y_val = train_test_split(
    X_tv, y_tv, test_size=0.25, random_state=42, stratify=y_tv
)

# The notebook's trial loop uses X_train_n / X_val_n / X_test_n by convention
# (previously meant "min-max normalized"); here we just alias them to the L2 data.
X_train_n = X_train
X_val_n   = X_val
X_test_n  = X_test

print(f"Train: {X_train_n.shape}  Val: {X_val_n.shape}  Test: {X_test_n.shape}")
print(f"Class balance (train): photon={(y_train==0).sum():,}  neutron={(y_train==1).sum():,}")
print(f"L2 norm sanity check (mean, should be ~1.0): {np.linalg.norm(X_train_n, axis=1).mean():.4f}")
print(f"Value range: [{X_train_n.min():.4f}, {X_train_n.max():.4f}]")

## Search space definition

We use **random search** rather than grid search — the full grid is over
100,000 combinations and grid would be wasteful when most hyperparameters
interact only weakly. Random search is the standard approach for high
dimensional spaces (Bergstra & Bengio 2012).

In [ ]:
# ── Search space (organized by which architecture each field applies to) ──

SHARED_SPACE = {
    "latent_dim":        [8],                # fixed to 8 per project decision
    "activation":        ["relu", "elu", "leaky_relu", "swish"],
    "output_activation": ["linear"],          # L2 data has negative values, no sigmoid
    "dropout":           [0.0, 0.1, 0.2, 0.3],
    "l2_reg":            [0.0, 1e-5, 1e-4, 1e-3],
    "batch_norm":        [False, True],
    "noise_std":         [0.0, 0.02, 0.05, 0.1],
    "optimizer":         ["adam", "adamw", "nadam"],
    "learning_rate":     [1e-4, 3e-4, 1e-3, 3e-3],
    "batch_size":        [256, 512, 1024],
    "loss":              ["mae", "mse", "huber"],
}

DENSE_SPACE = {
    "n_layers":       [1, 2, 3, 4, 5],
    "start_width":    [32, 64, 128, 256],
    "width_strategy": ["geometric", "linear", "funnel"],
}

CNN_SPACE = {
    "n_conv_layers":   [2, 3],
    "kernel_size":     [3, 5, 7],
    "n_filters_start": [16, 32, 64],
    "filter_strategy": ["double", "constant"],
}

MODEL_TYPES = ["dense", "cnn"]

N_TRIALS   = 100   # split roughly evenly between dense and cnn
MAX_EPOCHS = 25
PATIENCE   = 6

# How many unique configs are possible?
shared_combos = 1
for v in SHARED_SPACE.values():
    shared_combos *= len(v)
dense_combos = shared_combos
for v in DENSE_SPACE.values():
    dense_combos *= len(v)
cnn_combos = shared_combos
for v in CNN_SPACE.values():
    cnn_combos *= len(v)

print(f"Dense combos: {dense_combos:,}")
print(f"CNN combos:   {cnn_combos:,}")
print(f"Total:        {dense_combos + cnn_combos:,}")
print(f"Sampling {N_TRIALS} trials")

## Flexible model builder

Builds an autoencoder according to a hyperparameter dictionary. Supports all
the architecture/regularization options in the search space.

**Width computation:**
- `geometric` — halve the width at each layer (`start_width`, `start_width/2`, ...)
- `linear`    — evenly spaced from `start_width` down to `latent_dim`
- `funnel`    — random monotonically decreasing widths from a base set

In [ ]:
BASE_FUNNEL_WIDTHS = [256, 128, 64, 32, 16, 8]


# ── Helper functions (shared between dense and CNN) ─────────────────────────

def compute_widths(n_layers, start_width, latent_dim, strategy, rng):
    if strategy == "geometric":
        widths = []
        w = start_width
        for _ in range(n_layers):
            if w <= latent_dim:
                w = latent_dim * 2
            widths.append(int(w))
            w = max(latent_dim, w // 2)
        return widths
    elif strategy == "linear":
        end = max(latent_dim * 2, latent_dim + 4)
        if n_layers == 1:
            return [start_width]
        return np.linspace(start_width, end, n_layers).round().astype(int).tolist()
    elif strategy == "funnel":
        candidates = [w for w in BASE_FUNNEL_WIDTHS if w >= latent_dim]
        if not candidates:
            return [latent_dim * 2] * n_layers
        widths = []
        allowed = candidates.copy()
        for _ in range(n_layers):
            w = rng.choice(allowed)
            widths.append(int(w))
            allowed = [x for x in candidates if x <= w]
            if not allowed:
                allowed = [latent_dim * 2]
        return widths
    raise ValueError(f"Unknown strategy: {strategy}")


def get_activation(name):
    if name == "leaky_relu":
        return layers.LeakyReLU(negative_slope=0.1)
    return layers.Activation(name)


def get_optimizer(name, learning_rate):
    if name == "adam":  return keras.optimizers.Adam(learning_rate=learning_rate)
    if name == "adamw": return keras.optimizers.AdamW(learning_rate=learning_rate)
    if name == "nadam": return keras.optimizers.Nadam(learning_rate=learning_rate)
    raise ValueError(f"Unknown optimizer: {name}")


def get_loss(name):
    if name == "mae":   return "mae"
    if name == "mse":   return "mse"
    if name == "huber": return keras.losses.Huber()
    raise ValueError(f"Unknown loss: {name}")


# ── Dense autoencoder builder ───────────────────────────────────────────────

def build_dense_autoencoder(hp, n_features, rng):
    widths = compute_widths(
        hp["n_layers"], hp["start_width"], hp["latent_dim"], hp["width_strategy"], rng
    )
    reg = regularizers.l2(hp["l2_reg"]) if hp["l2_reg"] > 0 else None

    enc_inputs = layers.Input(shape=(n_features,), name="encoder_input")
    x = enc_inputs
    if hp["noise_std"] > 0:
        x = layers.GaussianNoise(hp["noise_std"])(x)
    for w in widths:
        x = layers.Dense(w, kernel_regularizer=reg)(x)
        if hp["batch_norm"]:
            x = layers.BatchNormalization()(x)
        x = get_activation(hp["activation"])(x)
        if hp["dropout"] > 0:
            x = layers.Dropout(hp["dropout"])(x)
    latent = layers.Dense(hp["latent_dim"], name="latent")(x)
    latent = get_activation(hp["activation"])(latent)
    encoder = Model(enc_inputs, latent, name="encoder")

    dec_inputs = layers.Input(shape=(hp["latent_dim"],), name="decoder_input")
    x = dec_inputs
    for w in reversed(widths):
        x = layers.Dense(w, kernel_regularizer=reg)(x)
        if hp["batch_norm"]:
            x = layers.BatchNormalization()(x)
        x = get_activation(hp["activation"])(x)
        if hp["dropout"] > 0:
            x = layers.Dropout(hp["dropout"])(x)
    out = layers.Dense(n_features)(x)
    out = layers.Activation(hp["output_activation"])(out)
    decoder = Model(dec_inputs, out, name="decoder")

    ae_inputs = layers.Input(shape=(n_features,))
    autoencoder = Model(ae_inputs, decoder(encoder(ae_inputs)), name="autoencoder")
    autoencoder.compile(
        optimizer=get_optimizer(hp["optimizer"], hp["learning_rate"]),
        loss=get_loss(hp["loss"]),
    )
    return autoencoder, encoder, {"widths": widths}


# ── 1D Convolutional autoencoder builder ────────────────────────────────────

def build_cnn_autoencoder(hp, n_features, rng):
    n_conv = hp["n_conv_layers"]
    k = hp["kernel_size"]
    f0 = hp["n_filters_start"]
    if hp["filter_strategy"] == "double":
        filters = [f0 * (2 ** i) for i in range(n_conv)]
    else:  # constant
        filters = [f0] * n_conv

    reg = regularizers.l2(hp["l2_reg"]) if hp["l2_reg"] > 0 else None

    # ── Encoder ──
    enc_inputs = layers.Input(shape=(n_features,), name="encoder_input")
    x = layers.Reshape((n_features, 1))(enc_inputs)
    if hp["noise_std"] > 0:
        x = layers.GaussianNoise(hp["noise_std"])(x)

    spatial = n_features
    for f in filters:
        x = layers.Conv1D(f, k, strides=2, padding="same", kernel_regularizer=reg)(x)
        if hp["batch_norm"]:
            x = layers.BatchNormalization()(x)
        x = get_activation(hp["activation"])(x)
        if hp["dropout"] > 0:
            x = layers.Dropout(hp["dropout"])(x)
        spatial = (spatial + 1) // 2  # ceil(spatial / 2) for stride 2 + same padding

    flat_size = spatial * filters[-1]
    x = layers.Flatten()(x)
    latent = layers.Dense(hp["latent_dim"], name="latent")(x)
    latent = get_activation(hp["activation"])(latent)
    encoder = Model(enc_inputs, latent, name="encoder")

    # ── Decoder (mirror) ──
    dec_inputs = layers.Input(shape=(hp["latent_dim"],), name="decoder_input")
    x = layers.Dense(flat_size)(dec_inputs)
    x = get_activation(hp["activation"])(x)
    x = layers.Reshape((spatial, filters[-1]))(x)

    # All but the last filter -- transposed convs that double the spatial dim
    for f in reversed(filters[:-1]):
        x = layers.Conv1DTranspose(f, k, strides=2, padding="same", kernel_regularizer=reg)(x)
        if hp["batch_norm"]:
            x = layers.BatchNormalization()(x)
        x = get_activation(hp["activation"])(x)
        if hp["dropout"] > 0:
            x = layers.Dropout(hp["dropout"])(x)

    # Final upsample back to (n_features, 1)
    x = layers.Conv1DTranspose(1, k, strides=2, padding="same", kernel_regularizer=reg)(x)
    x = layers.Activation(hp["output_activation"])(x)
    # The output may have an extra sample if n_features wasn't a power-of-2 multiple --
    # crop or pad to exactly n_features.
    out = layers.Lambda(lambda t: t[:, :n_features, 0], output_shape=(n_features,))(x)

    decoder = Model(dec_inputs, out, name="decoder")

    ae_inputs = layers.Input(shape=(n_features,))
    autoencoder = Model(ae_inputs, decoder(encoder(ae_inputs)), name="autoencoder")
    autoencoder.compile(
        optimizer=get_optimizer(hp["optimizer"], hp["learning_rate"]),
        loss=get_loss(hp["loss"]),
    )
    return autoencoder, encoder, {"filters": filters, "spatial": spatial}


# ── Dispatcher ──────────────────────────────────────────────────────────────

def build_model(hp, n_features=104, rng=None):
    rng = rng or random.Random(0)
    if hp["model_type"] == "dense":
        return build_dense_autoencoder(hp, n_features, rng)
    elif hp["model_type"] == "cnn":
        return build_cnn_autoencoder(hp, n_features, rng)
    raise ValueError(f"Unknown model_type: {hp['model_type']}")

## Trial evaluation function

For each trial we:
1. Train the autoencoder on the train split (with early stopping on val_loss)
2. Compute reconstruction loss on the test set
3. Extract latent features for train and test
4. Train a logistic regression on the latents
5. Evaluate classification accuracy on the test set
6. Compute a composite score combining both objectives

In [ ]:
# Maximum (val_loss - train_loss) / train_loss before a trial is flagged as overfit.
# 0.20 means val loss can be at most 20% higher than train loss.
OVERFIT_GAP_THRESHOLD = 0.20


def evaluate_classifier(encoder, X_train, y_train, X_eval, y_eval, batch_size=2048):
    # Train logistic regression on encoder features (extracted from train),
    # evaluate on the held-out X_eval / y_eval. During the search X_eval = val.
    # The test set is reserved and only used at the end for final evaluation.
    Z_train = encoder.predict(X_train, batch_size=batch_size, verbose=0)
    Z_eval  = encoder.predict(X_eval,  batch_size=batch_size, verbose=0)

    clf = make_pipeline(
        StandardScaler(),
        LogisticRegression(max_iter=2000, class_weight="balanced"),
    )
    clf.fit(Z_train, y_train)
    y_pred = clf.predict(Z_eval)

    acc = accuracy_score(y_eval, y_pred)
    cm = confusion_matrix(y_eval, y_pred, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel()
    photon_err  = fp / (tn + fp) if (tn + fp) > 0 else 0.0
    neutron_err = fn / (tp + fn) if (tp + fn) > 0 else 0.0
    return acc, photon_err, neutron_err, clf


def composite_score(val_loss, val_clf_acc, alpha=0.5):
    # Combined ranking metric (lower is better) - uses ONLY validation data.
    return alpha * val_loss + (1 - alpha) * (1 - val_clf_acc)


def run_trial(hp, X_train, X_val, y_train, y_val,
              max_epochs=MAX_EPOCHS, patience=PATIENCE, rng=None):
    # Build, train, and evaluate one config - test set is held out.
    keras.backend.clear_session()

    autoencoder, encoder, arch_info = build_model(hp, n_features=X_train.shape[1], rng=rng)
    n_params = autoencoder.count_params()

    callbacks = [
        keras.callbacks.EarlyStopping(
            monitor="val_loss", patience=patience,
            restore_best_weights=True, verbose=0,
        ),
        keras.callbacks.ReduceLROnPlateau(
            monitor="val_loss", patience=3, factor=0.5, verbose=0,
        ),
    ]

    t0 = time.time()
    history = autoencoder.fit(
        X_train, X_train,
        validation_data=(X_val, X_val),
        epochs=max_epochs,
        batch_size=hp["batch_size"],
        callbacks=callbacks,
        shuffle=True,
        verbose=0,
    )
    train_time = time.time() - t0

    # Train + val reconstruction loss after early stopping restored best weights
    train_loss = float(autoencoder.evaluate(X_train, X_train, batch_size=hp["batch_size"], verbose=0))
    val_loss   = float(autoencoder.evaluate(X_val,   X_val,   batch_size=hp["batch_size"], verbose=0))
    best_val   = float(np.min(history.history["val_loss"]))
    best_epoch = int(np.argmin(history.history["val_loss"]) + 1)

    # Per-trial overfit detection
    overfit_gap = (val_loss - train_loss) / max(train_loss, 1e-8)
    is_overfit = overfit_gap > OVERFIT_GAP_THRESHOLD

    # Classifier evaluated on VAL (not test) - composite_score uses only this
    val_clf_acc, val_photon_err, val_neutron_err, _ = evaluate_classifier(
        encoder, X_train, y_train, X_val, y_val
    )

    return {
        **hp,
        **arch_info,
        "n_params": n_params,
        "best_epoch": best_epoch,
        "train_loss": train_loss,
        "val_loss": val_loss,
        "best_val_loss": best_val,
        "overfit_gap": overfit_gap,
        "is_overfit": is_overfit,
        "val_clf_accuracy": val_clf_acc,
        "val_photon_err_pct": val_photon_err * 100,
        "val_neutron_err_pct": val_neutron_err * 100,
        "composite_score": composite_score(best_val, val_clf_acc),
        "train_time_s": round(train_time, 1),
    }


## Run the random search

Each trial trains an autoencoder, then a logistic regression on its latents.
Results are saved incrementally to `a1_tune_results.csv` so you
won't lose progress if interrupted.

In [ ]:
def sample_config(rng):
    # Pick model_type and only sample hyperparameters relevant to it.
    # Architecture-irrelevant fields are set to None so the CSV is consistent.
    model_type = rng.choice(MODEL_TYPES)
    cfg = {"model_type": model_type}
    for k, v in SHARED_SPACE.items():
        cfg[k] = rng.choice(v)

    if model_type == "dense":
        for k, v in DENSE_SPACE.items():
            cfg[k] = rng.choice(v)
        for k in CNN_SPACE:
            cfg[k] = None
    else:
        for k, v in CNN_SPACE.items():
            cfg[k] = rng.choice(v)
        for k in DENSE_SPACE:
            cfg[k] = None
    return cfg


def config_to_key(cfg):
    return tuple(sorted(cfg.items()))


rng = random.Random(42)
results = []

# Sample N_TRIALS unique configs
sampled_configs = []
seen = set()
while len(sampled_configs) < N_TRIALS:
    cfg = sample_config(rng)
    key = config_to_key(cfg)
    if key not in seen:
        seen.add(key)
        sampled_configs.append(cfg)

n_dense = sum(1 for c in sampled_configs if c["model_type"] == "dense")
n_cnn = sum(1 for c in sampled_configs if c["model_type"] == "cnn")
print(f"Sampled {len(sampled_configs)} configs: {n_dense} dense, {n_cnn} cnn")

for trial_num, hp in enumerate(sampled_configs):
    print(f"\n{'='*78}")
    print(f"Trial {trial_num+1}/{len(sampled_configs)}  [{hp['model_type'].upper()}]")
    if hp["model_type"] == "dense":
        print(f"  arch:  latent={hp['latent_dim']} layers={hp['n_layers']} start={hp['start_width']} "
              f"strategy={hp['width_strategy']} act={hp['activation']} out={hp['output_activation']}")
    else:
        print(f"  arch:  latent={hp['latent_dim']} conv_layers={hp['n_conv_layers']} kernel={hp['kernel_size']} "
              f"f0={hp['n_filters_start']} {hp['filter_strategy']} act={hp['activation']} out={hp['output_activation']}")
    print(f"  reg:   dropout={hp['dropout']} l2={hp['l2_reg']} bn={hp['batch_norm']} noise={hp['noise_std']}")
    print(f"  opt:   {hp['optimizer']} lr={hp['learning_rate']} bs={hp['batch_size']} loss={hp['loss']}")

    try:
        # NOTE: test set NOT passed in - it's held out for final evaluation only.
        result = run_trial(hp, X_train_n, X_val_n, y_train, y_val, rng=rng)
        results.append(result)
        flag = " [OVERFIT]" if result["is_overfit"] else ""
        print(f"  -> train={result['train_loss']:.5f}  val={result['val_loss']:.5f}  "
              f"gap={result['overfit_gap']:+.2%}{flag}  "
              f"clf_acc={result['val_clf_accuracy']:.4f}  "
              f"score={result['composite_score']:.5f}  "
              f"time={result['train_time_s']}s")
    except Exception as e:
        print(f"  -> FAILED: {e}")
        continue

    df = pd.DataFrame(results)
    df.to_csv("a1_tune_results.csv", index=False)

print(f"\n{'='*78}")
print(f"Complete: {len(results)} successful trials out of {len(sampled_configs)}")
n_overfit = sum(1 for r in results if r['is_overfit'])
print(f"Trials flagged as overfit: {n_overfit}")


## Results: top configurations

In [ ]:
df = pd.DataFrame(results)

print(f"Trials by model type:")
print(df["model_type"].value_counts().to_string())
print(f"\nOverfit trials excluded from rankings: {df['is_overfit'].sum()}")
print()

# Filter out overfit trials before ranking
clean_df = df[~df["is_overfit"]].copy()
print(f"Eligible trials: {len(clean_df)}")
print()

# ── Top 10 overall (composite score, val-based, no overfit) ──
print("=" * 90)
print("TOP 10 OVERALL (by composite score, overfit excluded)")
print("=" * 90)
shared_cols = ["model_type", "latent_dim", "activation", "output_activation",
               "dropout", "l2_reg", "batch_norm", "noise_std",
               "optimizer", "learning_rate", "batch_size", "loss",
               "train_loss", "val_loss", "overfit_gap", "val_clf_accuracy",
               "composite_score", "n_params"]
print(clean_df.sort_values("composite_score").head(10)[shared_cols].to_string())

# ── Top 5 dense ──
print("\n" + "=" * 90)
print("TOP 5 DENSE (overfit excluded)")
print("=" * 90)
dense_clean = clean_df[clean_df["model_type"] == "dense"]
if len(dense_clean) > 0:
    dense_cols = shared_cols + ["n_layers", "start_width", "width_strategy"]
    print(dense_clean.sort_values("composite_score").head(5)[dense_cols].to_string())

# ── Top 5 CNN ──
print("\n" + "=" * 90)
print("TOP 5 CNN (overfit excluded)")
print("=" * 90)
cnn_clean = clean_df[clean_df["model_type"] == "cnn"]
if len(cnn_clean) > 0:
    cnn_cols = shared_cols + ["n_conv_layers", "kernel_size", "n_filters_start", "filter_strategy"]
    print(cnn_clean.sort_values("composite_score").head(5)[cnn_cols].to_string())

# ── Architecture summary ──
print("\n" + "=" * 90)
print("ARCHITECTURE COMPARISON (mean across non-overfit trials)")
print("=" * 90)
summary = clean_df.groupby("model_type").agg(
    n_trials=("composite_score", "count"),
    mean_train_loss=("train_loss", "mean"),
    mean_val_loss=("val_loss", "mean"),
    mean_overfit_gap=("overfit_gap", "mean"),
    mean_clf_acc=("val_clf_accuracy", "mean"),
    best_composite=("composite_score", "min"),
    mean_params=("n_params", "mean"),
)
print(summary.round(5).to_string())


## Visualization: per-hyperparameter effects

These plots show how each individual hyperparameter affects classification
accuracy. Red dot = mean for that value. Use these to spot which values
consistently work and which to drop in a follow-up search.

In [ ]:
import os; os.makedirs("figures", exist_ok=True)
# ── Per-hyperparameter effect plots, colored by model type ──

shared_hps = [
    "latent_dim", "activation", "output_activation",
    "dropout", "l2_reg", "batch_norm", "noise_std",
    "optimizer", "learning_rate", "batch_size", "loss",
]

ncols = 4
nrows = int(np.ceil(len(shared_hps) / ncols))
fig, axes = plt.subplots(nrows, ncols, figsize=(4 * ncols, 3 * nrows))
axes = axes.flatten()

color_map = {"dense": "tab:blue", "cnn": "tab:orange"}

for ax, hp_name in zip(axes, shared_hps):
    for mt in ["dense", "cnn"]:
        sub = df[df["model_type"] == mt]
        if len(sub) == 0:
            continue
        vals = sub[hp_name].astype(str)
        ax.scatter(vals, sub["val_clf_accuracy"] * 100,
                   alpha=0.4, s=25, color=color_map[mt], label=mt)
    ax.set_title(hp_name, fontsize=10)
    ax.set_ylabel("Test acc [%]", fontsize=8)
    ax.tick_params(axis="x", rotation=30, labelsize=8)
    ax.grid(True, alpha=0.3)

axes[0].legend(fontsize=8)
for j in range(len(shared_hps), len(axes)):
    axes[j].set_visible(False)

fig.suptitle("Shared hyperparameter effects (dense=blue, cnn=orange)", fontsize=13)
plt.tight_layout()
plt.savefig("figures/a1_tune_hyperparameter_effects_shared.png", dpi=120, bbox_inches="tight")
plt.show()


# ── Dense-specific hyperparameters ──
dense_df = df[df["model_type"] == "dense"]
if len(dense_df) > 0:
    dense_hps = ["n_layers", "start_width", "width_strategy"]
    fig, axes = plt.subplots(1, 3, figsize=(12, 3))
    for ax, hp_name in zip(axes, dense_hps):
        vals = dense_df[hp_name].astype(str)
        unique = sorted(vals.unique())
        for v in unique:
            mask = vals == v
            ax.scatter([v] * mask.sum(),
                       dense_df.loc[mask, "val_clf_accuracy"].values * 100,
                       alpha=0.5, s=25, color="tab:blue")
        means = dense_df.groupby(vals)["val_clf_accuracy"].mean() * 100
        ax.scatter(means.index, means.values, color="tab:red", s=80, zorder=5, label="Mean")
        ax.set_title(hp_name)
        ax.set_ylabel("Test acc [%]")
        ax.grid(True, alpha=0.3)
    fig.suptitle("DENSE-only hyperparameter effects", fontsize=12)
    plt.tight_layout()
    plt.savefig("figures/a1_tune_hyperparameter_effects_dense.png", dpi=120, bbox_inches="tight")
    plt.show()


# ── CNN-specific hyperparameters ──
cnn_df = df[df["model_type"] == "cnn"]
if len(cnn_df) > 0:
    cnn_hps = ["n_conv_layers", "kernel_size", "n_filters_start", "filter_strategy"]
    fig, axes = plt.subplots(1, 4, figsize=(16, 3))
    for ax, hp_name in zip(axes, cnn_hps):
        vals = cnn_df[hp_name].astype(str)
        unique = sorted(vals.unique())
        for v in unique:
            mask = vals == v
            ax.scatter([v] * mask.sum(),
                       cnn_df.loc[mask, "val_clf_accuracy"].values * 100,
                       alpha=0.5, s=25, color="tab:orange")
        means = cnn_df.groupby(vals)["val_clf_accuracy"].mean() * 100
        ax.scatter(means.index, means.values, color="tab:red", s=80, zorder=5, label="Mean")
        ax.set_title(hp_name)
        ax.set_ylabel("Test acc [%]")
        ax.grid(True, alpha=0.3)
    fig.suptitle("CNN-only hyperparameter effects", fontsize=12)
    plt.tight_layout()
    plt.savefig("figures/a1_tune_hyperparameter_effects_cnn.png", dpi=120, bbox_inches="tight")
    plt.show()

## Pareto front: reconstruction vs classification

Models in the lower-right region (low reconstruction loss AND high
classification accuracy) are the best overall. Use this plot to spot the
tradeoff: pure reconstruction quality doesn't always give the best features
for classification.

In [ ]:
# Pareto plot: train vs val loss, with overfit trials marked
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: train_loss vs val_loss (overfit detection visualization)
ax = axes[0]
for mt, color in [("dense", "tab:blue"), ("cnn", "tab:orange")]:
    sub = df[df["model_type"] == mt]
    if len(sub) == 0:
        continue
    ok = sub[~sub["is_overfit"]]
    bad = sub[sub["is_overfit"]]
    ax.scatter(ok["train_loss"], ok["val_loss"], color=color, alpha=0.7,
               s=60, edgecolors="k", label=f"{mt} ok")
    if len(bad) > 0:
        ax.scatter(bad["train_loss"], bad["val_loss"], color=color, alpha=0.7,
                   s=80, edgecolors="red", linewidth=2, marker="X",
                   label=f"{mt} overfit")
lim = max(df.train_loss.max(), df.val_loss.max()) * 1.05
ax.plot([0, lim], [0, lim], "k--", alpha=0.5, label="y = x")
ax.plot([0, lim], [0, lim * (1 + OVERFIT_GAP_THRESHOLD)], "r--", alpha=0.5,
        label=f"y = x * {1+OVERFIT_GAP_THRESHOLD:.2f}")
ax.set_xlabel("Train reconstruction loss")
ax.set_ylabel("Val reconstruction loss")
ax.set_title("Train vs Val loss (overfit check)")
ax.set_xlim(0, lim)
ax.set_ylim(0, lim)
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)

# Right: val reconstruction vs val classification (the actual ranking)
ax = axes[1]
for mt, color in [("dense", "tab:blue"), ("cnn", "tab:orange")]:
    sub = df[(df["model_type"] == mt) & (~df["is_overfit"])]
    ax.scatter(sub["val_loss"], sub["val_clf_accuracy"] * 100,
               s=60, alpha=0.7, edgecolors="k", color=color, label=mt)
ax.set_xlabel("Val reconstruction loss")
ax.set_ylabel("Val classification accuracy [%]")
ax.set_title("Val reconstruction vs val classification (non-overfit only)")
ax.grid(True, alpha=0.3)
ax.legend()

# Highlight the best per architecture
clean_df = df[~df["is_overfit"]]
for mt, color in [("dense", "blue"), ("cnn", "darkorange")]:
    sub = clean_df[clean_df["model_type"] == mt]
    if len(sub) == 0:
        continue
    best = sub.loc[sub["composite_score"].idxmin()]
    ax.scatter([best["val_loss"]], [best["val_clf_accuracy"] * 100],
               marker="*", s=400, color=color, edgecolors="k",
               label=f"Best {mt}")

ax.legend()
plt.tight_layout()
plt.savefig("figures/a1_tune_pareto.png", dpi=120, bbox_inches="tight")
plt.show()


## Best configuration summary

In [ ]:
def print_config(label, row):
    print("=" * 70)
    print(f"BEST {label}")
    print("=" * 70)
    print(f"  model_type:        {row['model_type']}")
    print(f"  latent_dim:        {int(row['latent_dim'])}")
    if row["model_type"] == "dense":
        print(f"  n_layers:          {int(row['n_layers'])}")
        print(f"  start_width:       {int(row['start_width'])}")
        print(f"  width_strategy:    {row['width_strategy']}")
    else:
        print(f"  n_conv_layers:     {int(row['n_conv_layers'])}")
        print(f"  kernel_size:       {int(row['kernel_size'])}")
        print(f"  n_filters_start:   {int(row['n_filters_start'])}")
        print(f"  filter_strategy:   {row['filter_strategy']}")
    print(f"  activation:        {row['activation']}")
    print(f"  output_activation: {row['output_activation']}")
    print(f"  dropout:           {row['dropout']}")
    print(f"  l2_reg:            {row['l2_reg']}")
    print(f"  batch_norm:        {row['batch_norm']}")
    print(f"  noise_std:         {row['noise_std']}")
    print(f"  optimizer:         {row['optimizer']}")
    print(f"  learning_rate:     {row['learning_rate']}")
    print(f"  batch_size:        {int(row['batch_size'])}")
    print(f"  loss:              {row['loss']}")
    print()
    print(f"  n_params:          {int(row['n_params']):,}")
    print(f"  train_loss:        {row['train_loss']:.5f}")
    print(f"  val_loss:          {row['val_loss']:.5f}")
    print(f"  overfit_gap:       {row['overfit_gap']:+.2%}")
    print(f"  val_clf_accuracy:  {row['val_clf_accuracy']:.4f}")
    print(f"  composite_score:   {row['composite_score']:.5f}")
    print(f"  best_epoch:        {int(row['best_epoch'])}")
    print(f"  train_time_s:      {row['train_time_s']}")


def hp_dict_from_row(row):
    # Reconstruct the hyperparameter dict needed to rebuild the model
    keys = list(SHARED_SPACE) + list(DENSE_SPACE) + list(CNN_SPACE) + ["model_type"]
    return {k: (None if pd.isna(row[k]) else row[k]) for k in keys if k in row.index}


# ── Pick the best (overfit excluded) ──
clean_df = df[~df["is_overfit"]].copy()
best_overall = clean_df.loc[clean_df["composite_score"].idxmin()]

print_config("OVERALL (val-only)", best_overall)

best_dense_df = clean_df[clean_df["model_type"] == "dense"]
if len(best_dense_df) > 0:
    print()
    print_config("DENSE (val-only)", best_dense_df.loc[best_dense_df["composite_score"].idxmin()])

best_cnn_df = clean_df[clean_df["model_type"] == "cnn"]
if len(best_cnn_df) > 0:
    print()
    print_config("CNN (val-only)", best_cnn_df.loc[best_cnn_df["composite_score"].idxmin()])


# ── Honest test-set evaluation of the top-5 finalists ──
print("\n" + "=" * 70)
print("FINAL HONEST EVALUATION ON HELD-OUT TEST SET")
print("=" * 70)
print("Re-training the top-5 candidates from scratch and reporting on the")
print("test set, which was NOT used during the search at any point.")
print()

top5 = clean_df.sort_values("composite_score").head(5)
final_results = []

for rank, (_, row) in enumerate(top5.iterrows(), 1):
    hp = hp_dict_from_row(row)
    print(f"\nFinalist {rank}: {hp['model_type']} latent={int(hp['latent_dim'])} "
          f"composite={row['composite_score']:.5f} (val_acc={row['val_clf_accuracy']:.4f})")
    keras.backend.clear_session()
    rng2 = random.Random(rank * 1000)
    autoencoder, encoder, _ = build_model(hp, n_features=X_train_n.shape[1], rng=rng2)
    cb = [
        keras.callbacks.EarlyStopping(monitor="val_loss", patience=PATIENCE,
                                       restore_best_weights=True, verbose=0),
        keras.callbacks.ReduceLROnPlateau(monitor="val_loss", patience=3, factor=0.5, verbose=0),
    ]
    autoencoder.fit(X_train_n, X_train_n, validation_data=(X_val_n, X_val_n),
                    epochs=MAX_EPOCHS, batch_size=int(hp["batch_size"]),
                    callbacks=cb, shuffle=True, verbose=0)

    test_recon_loss = float(autoencoder.evaluate(X_test_n, X_test_n,
                                                  batch_size=int(hp["batch_size"]), verbose=0))
    test_clf_acc, test_photon, test_neutron, _ = evaluate_classifier(
        encoder, X_train_n, y_train, X_test_n, y_test
    )
    print(f"  test_recon_loss = {test_recon_loss:.5f}")
    print(f"  test_clf_acc    = {test_clf_acc:.4f}  "
          f"(val was {row['val_clf_accuracy']:.4f}, gap = {test_clf_acc - row['val_clf_accuracy']:+.4f})")
    print(f"  test photon err = {test_photon*100:.2f}%   neutron err = {test_neutron*100:.2f}%")
    final_results.append({
        "rank": rank,
        "model_type": hp["model_type"],
        "latent_dim": int(hp["latent_dim"]),
        "val_clf_acc": row["val_clf_accuracy"],
        "test_clf_acc": test_clf_acc,
        "val_test_gap": test_clf_acc - row["val_clf_accuracy"],
        "test_recon_loss": test_recon_loss,
    })

print("\n" + "=" * 70)
print("FINALIST SUMMARY")
print("=" * 70)
final_df = pd.DataFrame(final_results)
print(final_df.to_string(index=False))

df.to_csv("a1_tune_results.csv", index=False)
final_df.to_csv("a1_tune_finalists.csv", index=False)
print("\nResults saved to a1_tune_results.csv and a1_tune_finalists.csv")
